In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import mlflow

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (root_mean_squared_error, mean_absolute_error, r2_score)

In [2]:
df = pd.read_csv("C:\\Users\\awara\\Downloads\\MLops Day-01\\data\\Advertising.csv")
df.head()

,Unnamed: 0,TV,radio,newspaper,sales
0,1,230.1,37.8,69.2,22.1
1,2,44.5,39.3,45.1,10.4
2,3,17.2,45.9,69.3,9.3
3,4,151.5,41.3,58.5,18.5
4,5,180.8,10.8,58.4,12.9


In [3]:
X = df[['TV', 'radio', 'newspaper']]
y = df['sales']

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=56)

In [4]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")

In [5]:
mlflow.set_experiment("Advertising Sales Prediction")

<Experiment: artifact_location='file:c:/Users/awara/Downloads/MLops Day-01/notebooks/mlruns/1', creation_time=1788433274916, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1788433274916, lifecycle_stage='active', name='Advertising Sales Prediction', tags={}, trace_location=None, workspace='default'>

In [6]:
with mlflow.start_run(run_name="LinearRegression"):
    model = LinearRegression()
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    #Parameters
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 56)

    # Metrics
    mlflow.log_metric("test_rmse", rmse)
    mlflow.log_metric("test_r2", r2)

In [7]:
with mlflow.start_run(run_name="Ridge Regression"):
    model = Ridge(alpha=1.0)
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    #Parameters
    mlflow.log_param("model_type", "Ridge Regression")
    mlflow.log_param("alpha", 1.0)
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 56)

    # Metrics
    mlflow.log_metric("test_rmse", rmse)
    mlflow.log_metric("test_r2", r2)

    mlflow.sklearn.log_model(sk_model = model, name="Ridge_Reg_Model")

In [8]:
mlflow.sklearn.autolog()

In [9]:
with mlflow.start_run(run_name="Random Forest Autolog") as run:
    model = RandomForestRegressor(n_estimators=100, max_depth=5 ,random_state=42)
    model.fit(x_train, y_train)

    test_pred = model.predict(x_test)

    test_rmse = root_mean_squared_error(y_test, test_pred)
    test_r2 = r2_score(y_test, test_pred)
    test_mae = mean_absolute_error(y_test, test_pred)

    mlflow.log_metrics({
        "test_rmse": test_rmse,
        "test_r2": test_r2,
        "test_mae": test_mae
    })


2026/09/03 17:39:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [10]:
run_id = run.info.run_id
model_uri = f"runs:/{run_id}/model"
print(model_uri)

runs:/41e9e07073584f2382e45079e11f49cd/model


In [11]:
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name = "Advertising_Sales_Model"
)

registered_model 

Successfully registered model 'Advertising_Sales_Model'.
2026/09/03 17:54:26 WARNING mlflow.tracking._model_registry.fluent: Run with id 41e9e07073584f2382e45079e11f49cd has no artifacts at artifact path 'model', registering model based on models:/m-14cd1cdf9e4744b6ad7af9042e8630eb instead
Created version '1' of model 'Advertising_Sales_Model'.


<ModelVersion: aliases=[], creation_timestamp=1788438266239, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1788438266239, metrics=None, model_id=None, name='Advertising_Sales_Model', params=None, run_id='41e9e07073584f2382e45079e11f49cd', run_link=None, source='models:/m-14cd1cdf9e4744b6ad7af9042e8630eb', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>

In [12]:
from mlflow import MlflowClient

client = MlflowClient()

client.set_registered_model_alias(
    name="Advertising_Sales_Model",
    alias="champion",
    version="1"
)

In [13]:
model = mlflow.sklearn.load_model(
    "models:/Advertising_Sales_Model@champion"
)

new_data = pd.DataFrame({
    'TV':[150.0],
    'radio':[25.0],
    'newspaper':[30.0]
})

print(model.predict(new_data))

[15.13020624]
